In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y ffmpeg zstd

!pip install -q dspy-ai
!pip install -q python-telegram-bot
!pip install -q openai-whisper
!pip install -q ffmpeg-python
!pip install -q nest_asyncio


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import threading
import subprocess
import time
import requests

def run_ollama():
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

thread = threading.Thread(target=run_ollama)
thread.start()

time.sleep(5)

print("Ollama iniciado")

In [ ]:
!ollama pull qwen2.5:3b

In [ ]:
import dspy
import sqlite3
import requests
import subprocess
import time
import os
import whisper
import nest_asyncio

from telegram import Update
from telegram.ext import (
    ApplicationBuilder,
    MessageHandler,
    ContextTypes,
    filters
)


In [ ]:
def setup_ollama(model_name):
    try:
        requests.get(
            "http://localhost:11434/api/tags",
            timeout=5
        )

        print(f"Ollama rodando com {model_name}")
        return True

    except Exception as e:
        print("Ollama não está rodando")
        print(e)
        return False

In [ ]:
def create_streaming_database():

    conn = sqlite3.connect("streaming.db")
    c = conn.cursor()

    c.execute("DROP TABLE IF EXISTS reviews")
    c.execute("DROP TABLE IF EXISTS views")
    c.execute("DROP TABLE IF EXISTS users")
    c.execute("DROP TABLE IF EXISTS movies")

    c.execute("""
    CREATE TABLE movies (
        id INTEGER PRIMARY KEY,
        title TEXT,
        genre TEXT,
        release_year INTEGER,
        director TEXT,
        duration_min INTEGER,
        rating REAL
    )
    """)

    c.execute("""
    CREATE TABLE users (
        id INTEGER PRIMARY KEY,
        name TEXT,
        country TEXT,
        plan TEXT,
        join_date TEXT
    )
    """)

    c.execute("""
    CREATE TABLE views (
        id INTEGER PRIMARY KEY,
        user_id INTEGER,
        movie_id INTEGER,
        watch_date TEXT,
        completed INTEGER
    )
    """)

    c.execute("""
    CREATE TABLE reviews (
        id INTEGER PRIMARY KEY,
        user_id INTEGER,
        movie_id INTEGER,
        score INTEGER,
        comment TEXT
    )
    """)

    c.executemany(
        "INSERT INTO movies VALUES (?, ?, ?, ?, ?, ?, ?)",
        [
            (1, "Parasite", "Thriller", 2019, "Bong Joon-ho", 130, 8.5),
            (2, "Oldboy", "Thriller", 2003, "Park Chan-wook", 120, 8.4),
            (3, "Train to Busan", "Action", 2016, "Yeon Sang-ho", 118, 7.6),
            (4, "A Tale of Two Sisters", "Horror", 2003, "Kim Jee-woon", 115, 7.8),
            (5, "The Wailing", "Horror", 2016, "Na Hong-jin", 156, 7.9),
            (6, "Burning", "Drama", 2018, "Lee Chang-dong", 148, 7.5),
            (7, "Memories of Murder", "Crime", 2003, "Bong Joon-ho", 132, 8.1),
            (8, "Twinkling Watermelon", "Romance", 2023, "Son Jeong-hyun", 140, 8.3),
            (9, "My Sassy Girl", "Romance", 2001, "Kwak Jae-yong", 137, 7.8),
            (10, "The Man from Nowhere", "Action", 2010, "Lee Jeong-beom", 119, 7.8),
        ]
    )

    c.executemany(
        "INSERT INTO users VALUES (?, ?, ?, ?, ?)",
        [
            (1, "Marta", "Brazil", "Premium", "2022-01-10"),
            (2, "Emily", "USA", "Basic", "2021-05-20"),
            (3, "Wendy", "Brazil", "Premium", "2023-03-15"),
            (4, "Webster", "Canada", "Basic", "2022-08-01"),
            (5, "Ryan", "Brazil", "Premium", "2020-11-25"),
        ]
    )

    c.executemany(
        "INSERT INTO views VALUES (?, ?, ?, ?, ?)",
        [
            (1, 1, 1, "2024-01-05", 1),
            (2, 1, 3, "2024-01-10", 1),
            (3, 2, 5, "2024-01-12", 0),
            (4, 3, 2, "2024-02-01", 1),
            (5, 4, 7, "2024-02-10", 1),
            (6, 5, 4, "2024-03-05", 1),
            (7, 1, 6, "2024-03-10", 1),
            (8, 2, 8, "2024-03-15", 0),
            (9, 3, 9, "2024-04-01", 1),
            (10, 5, 10, "2024-04-10", 1),
        ]
    )

    c.executemany(
        "INSERT INTO reviews VALUES (?, ?, ?, ?, ?)",
        [
            (1, 1, 1, 9, "Crítica social muito boa!"),
            (2, 1, 3, 10, "Adorei!"),
            (3, 2, 5, 8, "Assustador"),
            (4, 3, 2, 10, "Clássico do cinema coreano"),
            (5, 4, 7, 9, "Investigação incrível"),
            (6, 5, 4, 9, "Me deixou com medo por dias"),
        ]
    )

    conn.commit()
    conn.close()

    print("Banco criado")

In [ ]:
create_streaming_database()

In [ ]:
class GenerateSQL(dspy.Signature):
    """
    Gere apenas queries SQL válidas para SQLite.
    Não explique.
    Não use markdown.
    Retorne apenas a query SQL.
    """

    question: str = dspy.InputField(
        desc="Pergunta do usuário"
    )

    sql_query: str = dspy.OutputField(
        desc="Query SQL válida"
    )


class RefineSQL(dspy.Signature):
    """
    Corrija queries SQL com erro.
    Retorne apenas SQL válido.
    """

    question: str = dspy.InputField()

    sql_query: str = dspy.InputField()

    error: str = dspy.InputField()

    refined_sql: str = dspy.OutputField(
        desc="SQL corrigido"
    )


class GenerateAnswer(dspy.Signature):
    """
    Gere respostas curtas, naturais e diretas.

    Exemplos:
    - "Existem 2 filmes de Horror."
    - "O filme com maior rating é Parasite."
    - "Marta, Wendy e Ryan possuem plano Premium."

    Não explique demais.
    Não invente informações.
    Seja breve.
    """

    question = dspy.InputField(
        desc="Pergunta original"
    )

    results = dspy.InputField(
        desc="Resultado da query SQL"
    )

    answer = dspy.OutputField(
        desc="Resposta curta e direta"
    )


print("Signatures definidas")

In [ ]:
class SQLGeneratorModule(dspy.Module):

    def __init__(self):
        super().__init__()

        self.generator = dspy.ChainOfThought(GenerateSQL)

        self.refiner = dspy.ChainOfThought(RefineSQL)

        self.answer_generator = dspy.ChainOfThought(
            GenerateAnswer
        )

    def clean_sql(self, sql):

        sql = sql.strip()

        if "```sql" in sql:
            sql = sql.split("```sql")[1].split("```")[0]

        elif "```" in sql:
            sql = sql.split("```")[1].split("```")[0]

        return sql.strip()

    def forward(self, question):

        db = sqlite3.connect("streaming.db")

        try:

            generated = self.generator(
                question=question
            )

            sql = self.clean_sql(
                generated.sql_query
            )

            results = db.execute(
                sql
            ).fetchall()

        except Exception as e:

            try:

                refined = self.refiner(
                    question=question,
                    sql_query=sql,
                    error=str(e)
                )

                sql = self.clean_sql(
                    refined.refined_sql
                )

                results = db.execute(
                    sql
                ).fetchall()

            except Exception as e2:

                db.close()

                return dspy.Prediction(
                    sql_query=sql,
                    results=None,
                    answer=None,
                    error=str(e2)
                )

        db.close()

        try:

            natural_answer = self.answer_generator(
                question=question,
                results=str(results)
            ).answer

            natural_answer = (
                natural_answer
                .replace("[[## completed ##]]", "")
                .replace("## completed ##", "")
                .strip()
            )

        except:
            natural_answer = str(results)

        return dspy.Prediction(
            sql_query=sql,
            results=results,
            answer=natural_answer,
            error=None
        )

In [ ]:
def create_examples():

    return [

        dspy.Example(
            question="Quantos filmes de Horror existem?",
            sql_query="SELECT COUNT(*) FROM movies WHERE genre = 'Horror'"
        ).with_inputs("question"),

        dspy.Example(
            question="Qual o filme com maior rating?",
            sql_query="SELECT title, rating FROM movies ORDER BY rating DESC LIMIT 1"
        ).with_inputs("question"),

        dspy.Example(
            question="Usuários Premium do Brasil",
            sql_query="SELECT name FROM users WHERE country = 'Brazil' AND plan = 'Premium'"
        ).with_inputs("question"),
    ]

examples = create_examples()

In [ ]:
def build_optimized_module():

    base_module = SQLGeneratorModule()

    optimizer = dspy.LabeledFewShot(k=3)

    optimized = optimizer.compile(
        student=base_module,
        trainset=examples
    )

    return optimized

In [ ]:
dspy.settings.configure(
    lm=dspy.LM(
        model="ollama/qwen2.5:3b",
        api_base="http://localhost:11434",
        temperature=0.2,
        max_tokens=100
    )
)

telegram_agent = build_optimized_module()

print("Modelo configurado")

In [ ]:
print("Carregando Whisper...")

whisper_model = whisper.load_model("base")

print("Whisper carregado")

In [ ]:
TOKEN = "8946890161:AAHoQ8Uu7vCum_gAEAyrfjRGbeCgwKNRG38"

In [ ]:
def format_results(results):

    if not results:
        return "Nenhum resultado encontrado."

    if len(results) == 1 and len(results[0]) == 1:
        return str(results[0][0])

    text = ""

    for row in results:
        text += " | ".join(
            str(item) for item in row
        ) + "\n"

    return text.strip()

In [ ]:
async def handle_text(
    update: Update,
    context: ContextTypes.DEFAULT_TYPE
):

    question = update.message.text

    result = telegram_agent(
        question=question
    )

    if result.error:

        await update.message.reply_text(
            f"Erro:\n{result.error}"
        )

        return

    await update.message.reply_text(
        result.answer
    )

In [ ]:
async def handle_voice(
    update: Update,
    context: ContextTypes.DEFAULT_TYPE
):

    voice = update.message.voice

    file = await context.bot.get_file(
        voice.file_id
    )

    await file.download_to_drive(
        "audio.ogg"
    )

    subprocess.run([
        "ffmpeg",
        "-y",
        "-i",
        "audio.ogg",
        "audio.wav"
    ])

    transcription = whisper_model.transcribe(
        "audio.wav"
    )

    question = transcription["text"]

    result = telegram_agent(
        question=question
    )

    if result.error:

        await update.message.reply_text(
            f"Erro:\n{result.error}"
        )

        return

    await update.message.reply_text(
        result.answer
    )

In [ ]:
app = ApplicationBuilder().token(
    TOKEN
).build()

app.add_handler(
    MessageHandler(
        filters.TEXT & ~filters.COMMAND,
        handle_text
    )
)

app.add_handler(
    MessageHandler(
        filters.VOICE,
        handle_voice
    )
)

print("Bot configurado")

In [ ]:
nest_asyncio.apply()

print("Bot online")

await app.initialize()
await app.start()
await app.updater.start_polling()